# CS 521 HW3, Problem 1: PGD Adversarial Training

Run on a Colab T4 GPU. Built on the Problem 3 starter; models are trained from scratch for 5 epochs.

# Setup

Download the CIFAR-10 train subset (10k images) and test subset (1k images).

In [1]:
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Downloading...
From (original): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe
From (redirected): https://drive.google.com/uc?id=1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe&confirm=t&uuid=fdaa737b-800a-494b-b164-031ad7a73caa
To: /content/cifar10_train_10k.pt
100% 30.8M/30.8M [00:00<00:00, 77.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /content/cifar10_test_1k.pt
100% 3.08M/3.08M [00:00<00:00, 22.7MB/s]


In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [5]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Attacks

Untargeted PGD for $\ell_\infty$ and $\ell_2$ (from HW1) with a random start; the $\ell_1$ attack is from the starter.

In [6]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = (torch.rand_like(x) * 2 - 1) * eps if rand_init else torch.zeros_like(x)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad.sign()
        delta = delta.clamp(-eps, eps)
        delta = clip_box(x, delta)
    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.randn_like(x)
        delta = delta / l2_norm(delta).clamp_min(1e-12) * eps * torch.rand(x.shape[0], 1, 1, 1, device=x.device)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad / (l2_norm(grad) + 1e-12)
        delta = delta * (eps / l2_norm(delta).clamp_min(eps))
        delta = clip_box(x, delta)
    return (x + delta).detach()

In [7]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluation

`eval_linf` returns clean and PGD (50 steps, 2 restarts) accuracy; `fgsm=True` runs a single-step attack.

In [8]:
def eval_linf(model, eps, steps=50, restarts=2, batch_size=250, fgsm=False):
    model.eval()
    torch.manual_seed(0)
    n_clean, n_rob = 0, 0
    for x, y in batches(x_test, y_test, batch_size, shuffle=False):
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        robust = clean.clone()
        if eps > 0:
            for _ in range(1 if fgsm else restarts):
                if fgsm:
                    x_adv = pgd_linf(model, x, y, eps, alpha=eps, steps=1, rand_init=False)
                else:
                    x_adv = pgd_linf(model, x, y, eps, alpha=2.5 * eps / steps, steps=steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
        n_clean += clean.sum().item()
        n_rob += robust.sum().item()
    n = len(y_test)
    return {"clean": 100.0 * n_clean / n, "robust": 100.0 * n_rob / n}

# Adversarial Training

`adv_train` trains for `epochs` epochs on the given loss (5 epochs here).

In [9]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Standard and $\ell_\infty$ Adversarial Training

Train both models from scratch.

In [10]:
def standard_loss(model, x, y):
    model.train()
    return F.cross_entropy(model(x), y)


def single_norm_loss(model, x, y, norm):
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)


def linf_loss_eps(model, x, y, eps, steps=10):
    model.eval()
    x_adv = pgd_linf(model, x, y, eps, alpha=2.5 * eps / steps, steps=steps)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [11]:
EPOCHS = 5

model_std = adv_train(make_model(), standard_loss, epochs=EPOCHS)
print("standard model (eps = 8/255):", eval_linf(model_std, 8 / 255))

epoch 1: loss 1.898 (8 s)
epoch 2: loss 1.561 (13 s)
epoch 3: loss 1.367 (17 s)
epoch 4: loss 1.205 (22 s)
epoch 5: loss 1.122 (27 s)
standard model (eps = 8/255): {'clean': 58.7, 'robust': 0.0}


In [12]:
model_adv8 = adv_train(make_model(), single_norm_loss, epochs=EPOCHS, norm="linf")
print("Linf adv. trained (eps = 8/255):", eval_linf(model_adv8, 8 / 255))

epoch 1: loss 2.183 (40 s)
epoch 2: loss 2.102 (82 s)
epoch 3: loss 2.053 (124 s)
epoch 4: loss 2.030 (165 s)
epoch 5: loss 2.015 (207 s)
Linf adv. trained (eps = 8/255): {'clean': 37.3, 'robust': 23.7}


### Different training $\epsilon$

Train with $\epsilon$ = 2/255, 4/255 and 16/255, then evaluate every model.

In [13]:
models = {"standard": model_std, "adv eps=8/255": model_adv8}
for e in (2, 4, 16):
    models[f"adv eps={e}/255"] = adv_train(make_model(), linf_loss_eps, epochs=EPOCHS, eps=e / 255)

epoch 1: loss 1.997 (42 s)
epoch 2: loss 1.799 (84 s)
epoch 3: loss 1.716 (125 s)
epoch 4: loss 1.671 (167 s)
epoch 5: loss 1.638 (208 s)
epoch 1: loss 2.077 (41 s)
epoch 2: loss 1.933 (83 s)
epoch 3: loss 1.867 (124 s)
epoch 4: loss 1.836 (166 s)
epoch 5: loss 1.815 (207 s)
epoch 1: loss 2.292 (42 s)
epoch 2: loss 2.244 (83 s)
epoch 3: loss 2.221 (125 s)
epoch 4: loss 2.213 (166 s)
epoch 5: loss 2.205 (208 s)


In [15]:
import pandas as pd

EPS_LIST = (0, 1, 2, 4, 8, 16)
rows = {}
for name, m in models.items():
    rows[name] = {f"{e}/255": eval_linf(m, e / 255)["robust"] for e in EPS_LIST}
pgd_acc = pd.DataFrame(rows).T
table_a = pgd_acc.round(1)
table_a.columns = ["clean" if c == "0/255" else f"PGD-50 eps={c}" for c in table_a.columns]
table_a

,clean,PGD-50 eps=1/255,PGD-50 eps=2/255,PGD-50 eps=4/255,PGD-50 eps=8/255,PGD-50 eps=16/255
standard,58.7,38.2,23.3,5.9,0.0,0.0
adv eps=8/255,37.3,35.5,33.5,30.4,23.7,12.3
adv eps=2/255,47.2,42.1,38.0,29.7,16.3,2.7
adv eps=4/255,41.7,38.1,35.8,30.7,20.5,7.2
adv eps=16/255,28.5,27.9,27.2,26.3,23.1,18.5


# Part (b): Attacking the Adversarially Trained Model

Compare PGD and FGSM across attack $\epsilon$ for every model.

In [16]:
def drop(m, eps):
    r = eval_linf(m, eps)
    return r["clean"], r["robust"], r["clean"] - r["robust"]

print(f"{'model':16s}{'clean':>8s}{'PGD-50 @8/255':>16s}{'drop':>8s}")
for name in ("standard", "adv eps=8/255"):
    c, r, d = drop(models[name], 8 / 255)
    print(f"{name:16s}{c:8.1f}{r:16.1f}{d:8.1f}")

model              clean   PGD-50 @8/255    drop
standard            58.7             0.0    58.7
adv eps=8/255       37.3            23.7    13.6


In [17]:
rows_f = {}
for name, m in models.items():
    rows_f[name] = {f"{e}/255": eval_linf(m, e / 255, fgsm=True)["robust"] for e in EPS_LIST[1:]}
fgsm_acc = pd.DataFrame(rows_f).T
pgd_sub = pgd_acc[[f"{e}/255" for e in EPS_LIST[1:]]]
print("FGSM accuracy (%)");    display(fgsm_acc.round(1))
print("PGD-50 accuracy (%)");  display(pgd_sub.round(1))

FGSM accuracy (%)


,1/255,2/255,4/255,8/255,16/255
standard,39.9,26.9,13.4,3.1,0.3
adv eps=8/255,35.6,33.6,30.4,24.3,14.8
adv eps=2/255,42.2,38.5,30.8,20.0,7.6
adv eps=4/255,38.1,35.8,30.9,22.2,11.9
adv eps=16/255,27.9,27.2,26.5,23.5,19.7


PGD-50 accuracy (%)


,1/255,2/255,4/255,8/255,16/255
standard,38.2,23.3,5.9,0.0,0.0
adv eps=8/255,35.5,33.5,30.4,23.7,12.3
adv eps=2/255,42.1,38.0,29.7,16.3,2.7
adv eps=4/255,38.1,35.8,30.7,20.5,7.2
adv eps=16/255,27.9,27.2,26.3,23.1,18.5


In [18]:
gap = (fgsm_acc - pgd_sub).round(1)
print("FGSM acc - PGD acc (points)"); gap

FGSM acc - PGD acc (points)


,1/255,2/255,4/255,8/255,16/255
standard,1.7,3.6,7.5,3.1,0.3
adv eps=8/255,0.1,0.1,0.0,0.6,2.5
adv eps=2/255,0.1,0.5,1.1,3.7,4.9
adv eps=4/255,0.0,0.0,0.2,1.7,4.7
adv eps=16/255,0.0,0.0,0.2,0.4,1.2


### Observations

(a) Standard training gives 58.7% clean and 0.0% PGD accuracy at 8/255; adversarial training at 8/255 gives 37.3% clean and 23.7% robust, so robustness costs clean accuracy. A larger training epsilon lowers clean accuracy and raises robustness to large attacks; 8/255 is the best trade-off here and 16/255 underfits in 5 epochs.

(b) PGD drops the standard model by 58.7 points but the adversarially trained model by only 13.6. FGSM is almost as strong as PGD against it (24.3% vs 23.7% at 8/255) because adversarial training makes the loss locally linear, while on the standard model FGSM is clearly weaker (13.4% vs 5.9% at 4/255). The FGSM-PGD gap grows once the attack epsilon exceeds the training epsilon, and there is no sign of gradient masking.

With 1,000 test images each accuracy has about 1.5 points of standard error, so differences under 1-2 points are noise.